# Time Series Forecasting

## Task Description
Perform zero-shot time series forecasting using the Chronos model.

In this notebook, we will:
1. Install libraries.
2. Import modules.
3. Load and prepare data.
4. Data Cleaning (if needed).
5. Load Chronos model.
6. Inference (Forecast).
7. Save Results.
8. Advanced Usage.


## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source models and is optimized to run on the Google Colab Free Tier (T4 GPU).

## 1. Installation
Install `chronos-forecasting`, `pandas`, `matplotlib`, `torch`.

In [ ]:
!pip install -q git+https://github.com/amazon-science/chronos-forecasting.git pandas matplotlib torch

## 2. Imports
Import necessary libraries.

In [ ]:
import torch
from chronos import ChronosPipeline
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## 3. Load Data
We will create a synthetic time series dataset (Sine wave with noise).

In [ ]:
# Generate synthetic data
t = np.linspace(0, 20, 100)
data = np.sin(t) + np.random.normal(0, 0.1, 100)
df = pd.DataFrame({'timestamp': pd.date_range(start='2023-01-01', periods=100, freq='D'), 'value': data})
print(df.head())

## 4. Data Cleaning
Ensure data is in the correct format (tensor).

In [ ]:
context = torch.tensor(df['value'].values)
print(f"Context shape: {context.shape}")

## 5. Load Model
Load `amazon/chronos-t5-small`.

In [ ]:
pipeline_ts = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="cuda",
    torch_dtype=torch.bfloat16,
)
print("Model loaded.")

## 6. Inference
Forecast the next 20 steps.

In [ ]:
prediction_length = 20
forecast = pipeline_ts.predict(context, prediction_length)

# Visualization
low, median, high = np.quantile(forecast[0].numpy(), [0.1, 0.5, 0.9], axis=0)
forecast_index = range(len(df), len(df) + prediction_length)

plt.figure(figsize=(10, 5))
plt.plot(df.index, df['value'], color="black", label="History")
plt.plot(forecast_index, median, color="tomato", label="Median Forecast")
plt.fill_between(forecast_index, low, high, color="tomato", alpha=0.3, label="80% Prediction Interval")
plt.legend()
plt.show()

## 7. Save Results
Save the forecast.

In [ ]:
results = {'median_forecast': median.tolist(), 'low_80': low.tolist(), 'high_80': high.tolist()}
with open('results.txt', 'w') as f:
    f.write(str(results))
print('Results saved to results.txt')

## 8. Advanced Usage
Try using `amazon/chronos-t5-large` for better accuracy.

In [ ]:
print("Try larger models for complex data!")